In [1]:
import pandas as pd
import json
import os
from sentence_transformers import SentenceTransformer

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


# 1. Getting the tasks for every PSOC code

## a) Create a mapping where for each PSOC code, get the list of tasks

In the PSOC Excel file found in https://psa.gov.ph/classification/psoc, the **OCCUPATIONAL TITLES AND DEFINITIONS** field follows a consistent structure containing the occupation title, a brief description, a list of tasks, and additional occupation names.

Since the tasks are the primary information of interest, we extract them using the following pattern:

* Tasks begin after **"Their tasks include:"**
* Additional occupation names begin after **"Examples of the occupations classified here:"**

Thus, the text between these two markers is extracted as the occupation's task description.


In [2]:
filepath = '../data/labor_codes/2022-Updates-to-the-2012-PSOC.xlsx'
psoc_sheets = pd.read_excel(filepath, sheet_name=None, dtype={'UNIT\nGROUP':'str'})
psoc_df = pd.concat(psoc_sheets.values())
psoc_df['UNIT\nGROUP'] = psoc_df['UNIT\nGROUP'].ffill()

# Get only the code and the statements
psoc_code_statements = psoc_df[['UNIT\nGROUP', 'OCCUPATIONAL TITLES AND DEFINITIONS']].copy().dropna()
psoc_code_statements.columns = ['Code', 'Statements']
psoc_code_statements.Statements = psoc_code_statements.Statements.str.strip()

# Use the pattern to determine if it is a task or not
pattern = r'(Their tasks include:|Their tasks would include:|In such cases, tasks would include:|Examples of the occupations classified here:)'
psoc_code_statements['Type'] = (
    psoc_code_statements['Statements']
    .str.extract(pattern, expand=False)
    .map({
        'Their tasks include:': 'Task',
        'Their tasks would include:': 'Task',
        'In such cases, tasks would include:': 'Task',
        'Examples of the occupations classified here:': 'Occupations'
    })
)
psoc_code_statements.Type = psoc_code_statements.Type.ffill()

psoc_code_statements.dropna(inplace=True)

# get the actual tasks
psoc_tasks = psoc_code_statements[psoc_code_statements.Type == 'Task'].copy()
# remove the pesky 'Their tasks include:'
psoc_tasks['Is Actual Task'] = (
    psoc_tasks['Statements']
    .str.contains(r'\w\)')
)
# remove the pesky a) or b) or c)
psoc_tasks = psoc_tasks[psoc_tasks['Is Actual Task']]
psoc_tasks = psoc_tasks[['Code', 'Statements']].copy()
psoc_tasks.Statements = psoc_tasks.Statements.str.extract(r'\w\)\s(.*)')
psoc_tasks.dropna(inplace=True)

# b) Embed each task into a vector

In [8]:
def convert_df_dict(task_map_df):
    """
    Convert a DataFrame of codes and tasks into a dictionary mapping
    each code to a list of associated tasks.

    Parameters
    ----------
    task_map_df : pandas.DataFrame
        DataFrame containing two columns: a code and its corresponding task.

    Returns
    -------
    dict
        Dictionary where each code maps to a list of associated tasks.
    """
    task_map = {}
    for _, row in task_map_df.iterrows():
        code, task = row
        task_map.setdefault(code, []).append(task)
    return task_map


def embed_tasks(task_map_df, model='all-mpnet-base-v2', verbose=False):
    """
    Embed each task in a code-to-tasks dataframe using the
    all-mpnet-base-v2 sentence-transformer model.

    Parameters
    ----------
    task_map : DataFrame

    Returns
    -------
    dict
        A dictionary with the same occupation codes, where each
        value is a list of normalized embedding vectors corresponding
        to the tasks associated with that code.
    """

    task_map = convert_df_dict(task_map_df)

    embedding_model = SentenceTransformer(model)
    embedded_task_map = {code : [] for code, _ in task_map.items()}

    # for each PSOC code and its tasks, embed it
    for code, tasks in task_map.items():
        if verbose:
            print(code)
        for task in tasks:
            embedded_task = embedding_model.encode(
                task,
                normalize_embeddings=True
            )
            embedded_task_map[code].append(embedded_task)
    return embedded_task_map


The following PSOC Codes (3435, 8189, 7319, 3139) do not have tasks outlined by PSA. Hence, we will just use their descriptions.

In [9]:
filepath = '../data/auxiliary/psoc_tasks_map.json'

if os.path.exists(filepath):
    # Load psoc_tasks
    with open(filepath, 'r') as file:
        psoc_tasks_map = json.load(file)
else:
    # convert the dataframe into a dictionary and save it
    psoc_tasks_map = convert_df_dict(psoc_tasks)

    psoc_tasks_map.update(
        {
            '3435': ['This unit group covers artistic and cultural associate professionals not classified elsewhere in Minor Group 344, Artistic, cultural and culinary associate professionals. For instance those who assist directors or actors with staging of theatrical, motion picture, television or commercial productions are classified here.'],
            '8189': ['This unit group includes stationary plant and machine operators not classified elsewhere in sub-major group 81,  Stationary plant and machine operators. The group includes, for instance, operators of machines which make  silicon chips and splice cables and ropes.'],
            '7319': ['This unit group covers handicraft workers who perform traditional handicrafts not classified elsewhere. For instance, the group includes traditional handicraft workers in non-precious metals and stone.'],
            '3139': ['This unit group covers process control technicians not classified elsewhere in minor group 313 Process control technicians. For instance, the group includes those who operate multiple process control equipment in manufacturing assembly lines and paper and pulp production.']
        }
    )
    with open(filepath, 'w') as file:
        json.dump(psoc_tasks_map, file)

# 2. Getting the tasks for every 2019 SOC-O*NET code

In [5]:
# open the soc onet data and clean up the O*NET-SOC Code
soc_tasks = pd.read_excel(
    '../data/labor_codes/Task Statements.xlsx', 
    dtype={'O*NET-SOC Code':'str'}
)
soc_tasks = soc_tasks[['O*NET-SOC Code', 'Task']].copy()

In [6]:
filepath = '../data/auxiliary/embedded_soc_tasks_map.json'

if os.path.exists(filepath):
    # Load existing embeddings
    with open(filepath, 'r') as file:
        embedded_soc_tasks = json.load(file)
else:
    # Generate embeddings
    embedded_soc_tasks = embed_tasks(soc_tasks)

    # Convert NumPy arrays to lists for JSON serialization
    embedded_soc_tasks = {
        code: [vector.tolist() for vector in vectors]
        for code, vectors in embedded_soc_tasks.items()
    }

    with open(filepath, 'w') as file:
        json.dump(embedded_soc_tasks, file)